In [1]:
# Unzip the uploaded file into the specified directory
!unzip -o /content/chanel-json.zip

Archive:  /content/chanel-json.zip
   creating: chanel-json/
  inflating: chanel-json/fragrance1.json  
  inflating: chanel-json/fragrance2.json  
  inflating: chanel-json/fragrance3.json  
  inflating: chanel-json/fragrance4.json  
  inflating: chanel-json/fragrance5.json  
  inflating: chanel-json/fragrance6.json  
  inflating: chanel-json/fragrance7.json  
  inflating: chanel-json/fragrance8.json  
  inflating: chanel-json/fragrance9.json  
  inflating: chanel-json/makeup1.json  
  inflating: chanel-json/makeup2.json  
  inflating: chanel-json/makeup3.json  
  inflating: chanel-json/makeup4.json  
  inflating: chanel-json/makeup5.json  
  inflating: chanel-json/makeup6.json  
  inflating: chanel-json/makeup7.json  
  inflating: chanel-json/skincare1.json  
  inflating: chanel-json/skincare2.json  
  inflating: chanel-json/skincare3.json  
  inflating: chanel-json/skincare4.json  
  inflating: chanel-json/skincare5.json  


In [2]:
import json
import os
from glob import glob
from bs4 import BeautifulSoup


def extract_products_from_html(html_string, base_url="https://www.chanel.com"):
    soup = BeautifulSoup(html_string, "html.parser")
    results = []

    products = soup.find_all("article", class_="product")

    for p in products:
        try:
            product = {}

            # --- Product ID / SKU ---
            product_id = p.get("data-id")
            product["product_id"] = product_id
            product["sku"] = product_id  # same in this structure

            # --- Name ---
            name_tag = p.find("span", class_="txt-product__title")
            product["product_name"] = name_tag.get_text(strip=True) if name_tag else None

            # --- Price ---
            price_tag = p.find("p", class_="is-price")
            if price_tag:
                price_text = price_tag.get_text(strip=True)
                product["price"] = price_text.replace("AED", "").strip()
            else:
                product["price"] = None

            # --- Product URL ---
            link_tag = p.find("a", href=True)
            if link_tag:
                product["product_url"] = base_url + link_tag["href"]
            else:
                product["product_url"] = None

            # --- Image URL ---
            img_tag = p.find("img")
            if img_tag:
                product["image_url"] = img_tag.get("data-src")
            else:
                product["image_url"] = None

            # --- Category & Subcategory ---
            parent_div = p.find_parent("div", class_="product-grid__item")
            if parent_div:
                product["category"] = parent_div.get("data-product-category")
                product["sub_category"] = parent_div.get("data-product-subcategory")
            else:
                product["category"] = None
                product["sub_category"] = None

            # --- Brand ---
            # Not explicitly available → infer from image/title
            if img_tag and img_tag.get("alt"):
                product["brand"] = img_tag["alt"].split()[0]
            else:
                product["brand"] = None

            # --- Variant ---
            desc_tag = p.find("span", class_="js-ellipsis")
            product["variant"] = desc_tag.get_text(strip=True) if desc_tag else None

            # --- Availability / Quantity ---
            # Not available → assume in stock if button exists
            add_btn = p.find("button", class_="quickview-add-to-bag")
            product["quantity"] = 1 if add_btn else 0

            results.append(product)

        except Exception as e:
            print(f"Error parsing product: {e}")

    return results


def process_json_files(folder_path):
    all_products = []
    seen_ids = set()

    files = glob(os.path.join(folder_path, "*.json"))

    for file in files:
        print(f"Processing: {file}")

        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        html_string = data.get("productListPage")

        if not html_string:
            continue

        products = extract_products_from_html(html_string)

        for p in products:
            if p["product_id"] not in seen_ids:
                seen_ids.add(p["product_id"])
                all_products.append(p)

    return all_products


def save_to_json(data, output_file):
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    folder_path = "chanel-json"  # change this
    output_file = "products_clean.json"

    products = process_json_files(folder_path)
    save_to_json(products, output_file)

    print(f"\n✅ Extracted {len(products)} products")

Processing: chanel-json/fragrance7.json
Processing: chanel-json/makeup2.json
Processing: chanel-json/skincare3.json
Processing: chanel-json/skincare5.json
Processing: chanel-json/makeup4.json
Processing: chanel-json/skincare4.json
Processing: chanel-json/fragrance3.json
Processing: chanel-json/fragrance5.json
Processing: chanel-json/fragrance8.json
Processing: chanel-json/makeup1.json
Processing: chanel-json/fragrance1.json
Processing: chanel-json/makeup5.json
Processing: chanel-json/fragrance2.json
Processing: chanel-json/skincare2.json
Processing: chanel-json/makeup3.json
Processing: chanel-json/fragrance6.json
Processing: chanel-json/fragrance9.json
Processing: chanel-json/fragrance4.json
Processing: chanel-json/skincare1.json
Processing: chanel-json/makeup7.json
Processing: chanel-json/makeup6.json

✅ Extracted 459 products


In [3]:
import pandas as pd

df = pd.DataFrame(products)
display(df.head())

,product_id,sku,product_name,price,product_url,image_url,category,sub_category,brand,variant,quantity
0,121880,121880,ALLURE HOMME,205,https://www.chanel.com/ae-en/fragrance/p/12188...,"https://www.chanel.com/images/t_one//w_0.51,h_...",Bath and Body,Men,CHANEL,SOAP,1
1,121820,121820,ALLURE HOMME,425,https://www.chanel.com/ae-en/fragrance/p/12182...,"https://www.chanel.com/images/t_one//w_0.51,h_...",Bath and Body,Men,CHANEL,ALL-OVER SPRAY,1
2,121700,121700,ALLURE HOMME,210,https://www.chanel.com/ae-en/fragrance/p/12170...,"https://www.chanel.com/images/t_one//w_0.51,h_...",Bath and Body,Men,CHANEL,DEODORANT STICK,1
3,121270,121270,ALLURE HOMME,350,https://www.chanel.com/ae-en/fragrance/p/12127...,"https://www.chanel.com/images/t_one//w_0.51,h_...",Bath and Body,Men,CHANEL,AFTER SHAVE LOTION,1
4,121250,121250,ALLURE HOMME,300,https://www.chanel.com/ae-en/fragrance/p/12125...,"https://www.chanel.com/images/t_one//w_0.55,h_...",Bath and Body,Men,CHANEL,AFTER SHAVE MOISTURISER,1


In [ ]:
# Save the DataFrame to a CSV file
df.to_csv("products_clean.csv", index=False)

print("DataFrame successfully saved to products_clean.csv")

DataFrame successfully saved to products_clean.csv


In [4]:
import json
import os
from glob import glob

def extract_products_from_file(file_path):
    results = []

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    product_list = data.get("dataLayer", {}).get("productList", {})

    for product_key, product_value in product_list.items():
        try:
            products = (
                product_value
                .get("quickviewPopin", {})
                .get("ecommerce", {})
                .get("detail", {})
                .get("products", [])
            )

            for p in products:
                product = {}

                # Basic fields
                product["product_name"] = p.get("name")
                product["price"] = p.get("price")
                product["product_id"] = p.get("id")
                product["sku"] = p.get("sku_id")
                product["variant"] = p.get("variant")
                product["brand"] = p.get("brand")
                product["quantity"] = p.get("quantity")

                # Category parsing
                raw_category = p.get("category", "")
                category_parts = raw_category.split("/")

                product["category"] = category_parts[1] if len(category_parts) > 1 else None
                product["sub_category"] = category_parts[2] if len(category_parts) > 2 else None

                # Product URL
                page_url = (
                    product_value
                    .get("quickviewPopin", {})
                    .get("page")
                )
                product["product_url"] = page_url

                # Image URL (not in your sample → set None safely)
                product["image_url"] = None

                results.append(product)

        except Exception as e:
            print(f"Error processing product {product_key} in {file_path}: {e}")

    return results


def process_folder(folder_path):
    all_products = []
    seen_ids = set()

    files = glob(os.path.join(folder_path, "*.json"))

    for file in files:
        print(f"Processing: {file}")
        products = extract_products_from_file(file)

        for p in products:
            # Deduplicate using product_id
            if p["product_id"] not in seen_ids:
                seen_ids.add(p["product_id"])
                all_products.append(p)

    return all_products


def save_to_json(data, output_file):
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    folder_path = "chanel-json"   # <-- change this
    output_file = "clean_products2.json"

    products = process_folder(folder_path)
    save_to_json(products, output_file)

    print(f"\n✅ Extracted {len(products)} unique products")

Processing: chanel-json/fragrance7.json
Processing: chanel-json/makeup2.json
Processing: chanel-json/skincare3.json
Processing: chanel-json/skincare5.json
Processing: chanel-json/makeup4.json
Processing: chanel-json/skincare4.json
Processing: chanel-json/fragrance3.json
Processing: chanel-json/fragrance5.json
Processing: chanel-json/fragrance8.json
Processing: chanel-json/makeup1.json
Processing: chanel-json/fragrance1.json
Processing: chanel-json/makeup5.json
Processing: chanel-json/fragrance2.json
Processing: chanel-json/skincare2.json
Processing: chanel-json/makeup3.json
Processing: chanel-json/fragrance6.json
Processing: chanel-json/fragrance9.json
Processing: chanel-json/fragrance4.json
Processing: chanel-json/skincare1.json
Processing: chanel-json/makeup7.json
Processing: chanel-json/makeup6.json

✅ Extracted 459 unique products


In [5]:
import pandas as pd

# Load the clean_products2.json file into a DataFrame
df_clean = pd.read_json("clean_products2.json")

# Display the first 5 rows of the DataFrame
display(df_clean.head())

,product_name,price,product_id,sku,variant,brand,quantity,category,sub_category,product_url,image_url
0,coco body cream,440,p113940,113990,(not set)/150 g,fnb,1,fragrance,women,/ae-en/fragrance/p/113990/coco-body-cream/quic...,NaN
1,egoiste deodorant stick,210,p114700,114700,(not set)/60 g,fnb,1,fragrance,bath and body,/ae-en/fragrance/p/114700/egoiste-deodorant-st...,NaN
2,coco mademoiselle fragrance primer,540,p116690,116690,(not set)/100 ml,fnb,1,fragrance,women,/ae-en/fragrance/p/116690/coco-mademoiselle-fr...,NaN
3,coco mademoiselle body oil,565,p116700,116700,(not set)/200 ml,fnb,1,fragrance,women,/ae-en/fragrance/p/116700/coco-mademoiselle-bo...,NaN
4,coco mademoiselle silky body cream,440,p116790,116790,(not set)/150 g,fnb,1,fragrance,women,/ae-en/fragrance/p/116790/coco-mademoiselle-si...,NaN


In [6]:
# Clean the 'price' column to extract numeric values for AED
df_clean['price_aed'] = df_clean['price'].astype(str).str.replace('AED', '', regex=False).str.replace(',', '', regex=False)
df_clean['price_aed'] = pd.to_numeric(df_clean['price_aed'], errors='coerce')

# Convert AED to USD (1 AED = 0.27 USD)
df_clean['price_usd'] = df_clean['price_aed'] * 0.27

# Display the DataFrame with the new price columns
display(df_clean.head())

,product_name,price,product_id,sku,variant,brand,quantity,category,sub_category,product_url,image_url,price_aed,price_usd
0,coco body cream,440,p113940,113990,(not set)/150 g,fnb,1,fragrance,women,/ae-en/fragrance/p/113990/coco-body-cream/quic...,NaN,440,118.80
1,egoiste deodorant stick,210,p114700,114700,(not set)/60 g,fnb,1,fragrance,bath and body,/ae-en/fragrance/p/114700/egoiste-deodorant-st...,NaN,210,56.70
2,coco mademoiselle fragrance primer,540,p116690,116690,(not set)/100 ml,fnb,1,fragrance,women,/ae-en/fragrance/p/116690/coco-mademoiselle-fr...,NaN,540,145.80
3,coco mademoiselle body oil,565,p116700,116700,(not set)/200 ml,fnb,1,fragrance,women,/ae-en/fragrance/p/116700/coco-mademoiselle-bo...,NaN,565,152.55
4,coco mademoiselle silky body cream,440,p116790,116790,(not set)/150 g,fnb,1,fragrance,women,/ae-en/fragrance/p/116790/coco-mademoiselle-si...,NaN,440,118.80


In [ ]:
search_query = "eau de cologne les exclusifs de chanel – eau de toilette"
result = df_clean[df_clean['product_name'].str.contains(search_query, case=False, na=False)]

if not result.empty:
    print(f"Found {len(result)} product(s) matching '{search_query}':")
    display(result)
else:
    print(f"No product found matching '{search_query}'.")

In [8]:
df_clean['sku'] = df_clean['sku'].astype(str)
df['sku'] = df['sku'].astype(str)
df_clean = df_clean.merge(df[['sku', 'image_url']], on='sku', how='left')
display(df_clean.head())

,product_name,price,product_id,sku,variant,brand,quantity,category,sub_category,product_url,image_url_x,price_aed,price_usd,image_url_y
0,coco body cream,440,p113940,113990,(not set)/150 g,fnb,1,fragrance,women,/ae-en/fragrance/p/113990/coco-body-cream/quic...,NaN,440,118.80,"https://www.chanel.com/images/t_one//w_0.51,h_..."
1,egoiste deodorant stick,210,p114700,114700,(not set)/60 g,fnb,1,fragrance,bath and body,/ae-en/fragrance/p/114700/egoiste-deodorant-st...,NaN,210,56.70,"https://www.chanel.com/images/t_one//w_0.51,h_..."
2,coco mademoiselle fragrance primer,540,p116690,116690,(not set)/100 ml,fnb,1,fragrance,women,/ae-en/fragrance/p/116690/coco-mademoiselle-fr...,NaN,540,145.80,"https://www.chanel.com/images/t_one//w_0.51,h_..."
3,coco mademoiselle body oil,565,p116700,116700,(not set)/200 ml,fnb,1,fragrance,women,/ae-en/fragrance/p/116700/coco-mademoiselle-bo...,NaN,565,152.55,"https://www.chanel.com/images/t_one//w_0.61,h_..."
4,coco mademoiselle silky body cream,440,p116790,116790,(not set)/150 g,fnb,1,fragrance,women,/ae-en/fragrance/p/116790/coco-mademoiselle-si...,NaN,440,118.80,"https://www.chanel.com/images/t_one//w_0.51,h_..."


In [9]:
# Drop the specified columns
df_clean = df_clean.drop(columns=['price', 'image_url_x'])

# Rename 'image_url_y' to 'image_url'
df_clean = df_clean.rename(columns={'image_url_y': 'image_url'})

# Add base URL to 'product_url'
df_clean['product_url'] = "https://www.chanel.com/" + df_clean['product_url']

# Display the updated DataFrame
display(df_clean.head())

,product_name,product_id,sku,variant,brand,quantity,category,sub_category,product_url,price_aed,price_usd,image_url
0,coco body cream,p113940,113990,(not set)/150 g,fnb,1,fragrance,women,https://www.chanel.com//ae-en/fragrance/p/1139...,440,118.80,"https://www.chanel.com/images/t_one//w_0.51,h_..."
1,egoiste deodorant stick,p114700,114700,(not set)/60 g,fnb,1,fragrance,bath and body,https://www.chanel.com//ae-en/fragrance/p/1147...,210,56.70,"https://www.chanel.com/images/t_one//w_0.51,h_..."
2,coco mademoiselle fragrance primer,p116690,116690,(not set)/100 ml,fnb,1,fragrance,women,https://www.chanel.com//ae-en/fragrance/p/1166...,540,145.80,"https://www.chanel.com/images/t_one//w_0.51,h_..."
3,coco mademoiselle body oil,p116700,116700,(not set)/200 ml,fnb,1,fragrance,women,https://www.chanel.com//ae-en/fragrance/p/1167...,565,152.55,"https://www.chanel.com/images/t_one//w_0.61,h_..."
4,coco mademoiselle silky body cream,p116790,116790,(not set)/150 g,fnb,1,fragrance,women,https://www.chanel.com//ae-en/fragrance/p/1167...,440,118.80,"https://www.chanel.com/images/t_one//w_0.51,h_..."


In [10]:
# Save the DataFrame to a JSON file
df_clean.to_json("final_chanel_products.json", orient="records", indent=2)

print("DataFrame successfully saved to final_chanel_products.json")

DataFrame successfully saved to final_chanel_products.json


In [11]:
# Save the DataFrame to a CSV file
df_clean.to_csv("final_chanel_products.csv", index=False)

print("DataFrame successfully saved to final_chanel_products.csv")

DataFrame successfully saved to final_chanel_products.csv


In [13]:
import requests

url = "https://www.chanel.com/ae-en/fragrance/p/102420/paris-venise-les-eaux-de-chanel-eau-de-toilette-spray/"
output_filename = "chanel_product_page.html"

# Define headers to mimic a web browser
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

try:
    # Pass the headers with the request
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"Successfully fetched and saved HTML from {url} to {output_filename}")
except requests.exceptions.HTTPError as errh:
    print(f"Http Error: {errh}")
except requests.exceptions.ConnectionError as errc:
    print(f"Error Connecting: {errc}")
except requests.exceptions.Timeout as errt:
    print(f"Timeout Error: {errt}")
except requests.exceptions.RequestException as err:
    print(f"Something went wrong: {err}")

Successfully fetched and saved HTML from https://www.chanel.com/ae-en/fragrance/p/102420/paris-venise-les-eaux-de-chanel-eau-de-toilette-spray/ to chanel_product_page.html


In [14]:
from bs4 import BeautifulSoup

def extract_variants_from_html_snippet(html_snippet):
    soup = BeautifulSoup(html_snippet, "html.parser")
    variants = []

    # Find the unordered list that contains the variants
    variant_list = soup.find("ul", class_="dropdown__list")

    if variant_list:
        # Find all list items representing individual variants
        variant_items = variant_list.find_all("li", class_="dropdown__list__variants")

        for item in variant_items:
            variant_data = {}
            # Extract size
            size_tag = item.find("span", class_="dropdown_size_text")
            if size_tag:
                variant_data["size"] = size_tag.get_text(strip=True).replace("\u00a0", " ") # Replace non-breaking space

            # Extract price
            price_tag = item.find("span", class_="dropdown_size_price")
            if price_tag:
                variant_data["price"] = price_tag.get_text(strip=True)

            if variant_data: # Only add if some data was extracted
                variants.append(variant_data)
    return variants

# HTML sample provided by the user
html_sample = """<ul class=\"dropdown__list\" role=\"presentation\" data-test=\"lblAllVarients\"> <li role=\"menuitem\" tabindex=\"0\" class=\"dropdown__list__variants\" aria-current=\"true\" data-test=\"btnVarientByValue\"> <a href=\"/ae-en/fragrance/p/102420/paris-venise-les-eaux-de-chanel-eau-de-toilette-spray/\" class=\"js-dropdown-item dropdown__list__item\" data-value=\"102420\"> <span data-test=\"lstFits_ByFITNAME\" class=\"dropdown_size_text\">125&nbsp;ml </span> <span class=\"dropdown_size_price\">765 AED</span> </a> </li> <li role=\"menuitem\" tabindex=\"0\" class=\"dropdown__list__variants\" data-test=\"btnVarientByValue\"> <a href=\"/ae-en/fragrance/p/102620/paris-venise-les-eaux-de-chanel-eau-de-toilette-spray/\" class=\"js-dropdown-item dropdown__list__item\" data-value=\"102620\"> <span data-test=\"lstFits_ByFITNAME\" class=\"dropdown_size_text\">50&nbsp;ml </span> <span class=\"dropdown_size_price\">530 AED</span> </a> </li> </ul>"""

# Demonstrate the function
extracted_variants = extract_variants_from_html_snippet(html_sample)
print("Extracted Variants:")
for variant in extracted_variants:
    print(variant)

Extracted Variants:
{'size': '125 ml', 'price': '765 AED'}
{'size': '50 ml', 'price': '530 AED'}


In [15]:
import os
import json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse

# -----------------------------
# CONFIG
# -----------------------------
OUTPUT_FOLDER = "chanel_pages"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


# -----------------------------
# FETCH HTML
# -----------------------------
def fetch_and_save_html(url, sku):
    try:
        response = requests.get(url, headers=HEADERS, timeout=20)
        response.raise_for_status()

        filename = os.path.join(OUTPUT_FOLDER, f"{sku}.html")

        with open(filename, "w", encoding="utf-8") as f:
            f.write(response.text)

        return response.text

    except Exception as e:
        print(f"❌ Failed to fetch {url}: {e}")
        return None


# -----------------------------
# EXTRACT VARIANTS
# -----------------------------
def extract_variants(html):
    soup = BeautifulSoup(html, "html.parser")
    variants = []

    variant_list = soup.find("ul", class_="dropdown__list")

    if not variant_list:
        return variants

    items = variant_list.find_all("li", class_="dropdown__list__variants")

    for item in items:
        variant_data = {}

        size_tag = item.find("span", class_="dropdown_size_text")
        price_tag = item.find("span", class_="dropdown_size_price")
        link_tag = item.find("a", href=True)

        if size_tag:
            variant_data["size"] = size_tag.get_text(strip=True).replace("\xa0", " ")

        if price_tag:
            price_text = price_tag.get_text(strip=True)
            variant_data["price_aed"] = int(price_text.replace("AED", "").strip())

        if link_tag:
            variant_data["variant_url"] = "https://www.chanel.com" + link_tag["href"]

            # extract SKU from URL
            try:
                variant_data["sku"] = link_tag["href"].split("/p/")[1].split("/")[0]
            except:
                variant_data["sku"] = None

        if variant_data:
            variants.append(variant_data)

    return variants


# -----------------------------
# MAIN PROCESS
# -----------------------------
def enrich_products(input_json):
    enriched = []

    for product in input_json:
        print(f"Processing: {product['product_name']}")

        html = fetch_and_save_html(product["product_url"], product["sku"])

        if not html:
            product["variants"] = []
            enriched.append(product)
            continue

        variants = extract_variants(html)

        # attach variants
        product["variants"] = variants

        enriched.append(product)

    return enriched


# -----------------------------
# RUN
# -----------------------------
if __name__ == "__main__":

    # load your input JSON
    with open("/content/final_chanel_products.json", "r", encoding="utf-8") as f:
        products = json.load(f)

    enriched_products = enrich_products(products)

    # save updated JSON
    with open("final_chanel_products_enriched.json", "w", encoding="utf-8") as f:
        json.dump(enriched_products, f, indent=2, ensure_ascii=False)

    print("\n✅ Done! Output saved to output_enriched.json")

Processing: coco body cream
Processing: egoiste deodorant stick
Processing:  coco mademoiselle fragrance primer
Processing: coco mademoiselle body oil
Processing: coco mademoiselle silky body cream
Processing: coco mademoiselle fresh deodorant spray
Processing: coco mademoiselle gentle perfumed soap
Processing: coco mademoiselle moisturizing body lotion
Processing: coco mademoiselle foaming shower gel
Processing: coco mademoiselle hair perfume
Processing: pour monsieur deodorant stick
Processing: antaeus deodorant stick
Processing: gabrielle chanel  body oil
Processing: gabrielle chanel body cream
Processing: gabrielle chanel fragrance primer
Processing: gabrielle chanel hair mist
Processing: gabrielle chanel deodorant spray
Processing: gabrielle chanel moisturizing body lotion
Processing: gabrielle chanel foaming shower gel
Processing: allure homme after shave moisturizer
Processing: allure homme after shave lotion
Processing: allure homme deodorant stick
Processing: allure homme all-

KeyboardInterrupt: 

In [16]:
import os
import json
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

# -----------------------------
# CONFIG
# -----------------------------
OUTPUT_FOLDER = "chanel_pages"
MAX_WORKERS = 10  # adjust (10–20 is usually safe)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
})


# -----------------------------
# FETCH HTML
# -----------------------------
def fetch_and_save_html(url, sku):
    try:
        response = session.get(url, timeout=20)
        response.raise_for_status()

        filename = os.path.join(OUTPUT_FOLDER, f"{sku}.html")

        with open(filename, "w", encoding="utf-8") as f:
            f.write(response.text)

        return response.text

    except Exception as e:
        print(f"❌ Failed: {sku} → {e}")
        return None


# -----------------------------
# EXTRACT VARIANTS
# -----------------------------
def extract_variants(html):
    soup = BeautifulSoup(html, "html.parser")
    variants = []

    variant_list = soup.find("ul", class_="dropdown__list")
    if not variant_list:
        return variants

    items = variant_list.find_all("li", class_="dropdown__list__variants")

    for item in items:
        variant_data = {}

        size_tag = item.find("span", class_="dropdown_size_text")
        price_tag = item.find("span", class_="dropdown_size_price")
        link_tag = item.find("a", href=True)

        if size_tag:
            variant_data["size"] = size_tag.get_text(strip=True).replace("\xa0", " ")

        if price_tag:
            try:
                variant_data["price_aed"] = int(
                    price_tag.get_text(strip=True).replace("AED", "").strip()
                )
            except:
                variant_data["price_aed"] = None

        if link_tag:
            href = link_tag["href"]
            variant_data["variant_url"] = "https://www.chanel.com" + href

            try:
                variant_data["sku"] = href.split("/p/")[1].split("/")[0]
            except:
                variant_data["sku"] = None

        if variant_data:
            variants.append(variant_data)

    return variants


# -----------------------------
# PROCESS SINGLE PRODUCT
# -----------------------------
def process_product(product):
    sku = product.get("sku")
    url = product.get("product_url")

    print(f"Processing: {sku}")

    html = fetch_and_save_html(url, sku)

    if not html:
        product["variants"] = []
        return product

    variants = extract_variants(html)
    product["variants"] = variants

    return product


# -----------------------------
# MAIN PARALLEL PROCESS
# -----------------------------
def enrich_products_parallel(products):
    results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_product, p) for p in products]

        for future in as_completed(futures):
            try:
                result = future.result()
                results.append(result)
            except Exception as e:
                print(f"❌ Error in thread: {e}")

    return results


# -----------------------------
# RUN
# -----------------------------
if __name__ == "__main__":

    with open("/content/final_chanel_products.json", "r", encoding="utf-8") as f:
        products = json.load(f)

    enriched_products = enrich_products_parallel(products)

    with open("final_chanel_products_enriched.json", "w", encoding="utf-8") as f:
        json.dump(enriched_products, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Done! Processed {len(enriched_products)} products")

Processing: 113990
Processing: 114700
Processing: 116690
Processing: 116700
Processing: 116790
Processing: 116860
Processing: 116900
Processing: 116945
Processing: 116965
Processing: 116997
Processing: 117921
Processing: 118700
Processing: 120820
Processing: 120830
Processing: 120840
Processing: 120870
Processing: 120930
Processing: 120940Processing: 120960

Processing: 121250
Processing: 121270
Processing: 121700
Processing: 121820
Processing: 121880
Processing: 101096
Processing: 101118
Processing: 137880
Processing: 138177
Processing: 138830
Processing: 138854
Processing: 138855
Processing: 138857
Processing: 138858
Processing: 138859
Processing: 138860
Processing: 138861
Processing: 140570
Processing: 144270
Processing: 147140
Processing: 148847
Processing: 156716
Processing: 165083
Processing: 169060
Processing: 171542
Processing: 171838
Processing: 175174
Processing: 183802
Processing: 186362
Processing: 133040
Processing: 133070
Processing: 133510
Processing: 133520
Processing: 

In [17]:
!zip -r chanel_pages.zip /content/chanel_pages

  adding: content/chanel_pages/ (stored 0%)
  adding: content/chanel_pages/147555.html (deflated 97%)
  adding: content/chanel_pages/136110.html (deflated 97%)
  adding: content/chanel_pages/140895.html (deflated 97%)
  adding: content/chanel_pages/121460.html (deflated 97%)
  adding: content/chanel_pages/179137.html (deflated 97%)
  adding: content/chanel_pages/102040.html (deflated 97%)
  adding: content/chanel_pages/122320.html (deflated 97%)
  adding: content/chanel_pages/185272.html (deflated 97%)
  adding: content/chanel_pages/132212.html (deflated 97%)
  adding: content/chanel_pages/133650.html (deflated 97%)
  adding: content/chanel_pages/120930.html (deflated 97%)
  adding: content/chanel_pages/179387.html (deflated 97%)
  adding: content/chanel_pages/120360.html (deflated 97%)
  adding: content/chanel_pages/141390.html (deflated 97%)
  adding: content/chanel_pages/113660.html (deflated 97%)
  adding: content/chanel_pages/140025.html (deflated 97%)
  adding: content/chanel_pag